#### Phase 1: Loading Raw Data

##### imports

In [125]:
import pandas as pd
import numpy as np
import matplotlib as plb
import json
import kagglehub as kg
from kagglehub import KaggleDatasetAdapter
import os

###### JSON Path

In [126]:
LOCAL_JSON_PATH = '../data/raw/raw_pokemon.json'

In [ ]:
# The code in this cell checks if the data has already been downloaded and saved under the name raw_pokemon.json.
# If the data hasn't been found, it will be downloaded from Kaggle

if not os.path.exists(LOCAL_JSON_PATH):
    print("Local file not found. Downloading from Kaggle.")
    
    # 1. Define the exact columns we want to keep
    columns_to_keep = ['Number', ' Name', ' Type1', ' Type2', ' Base_Total', ' HP', ' Attack', ' Defense', ' Evolutions', 'Legendary']

    
    # 1. Download from Kaggle and turn it into a DataFrame
    df = kg.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "dizzypanda/gen-1-pokemon",
        "FirstGenPokemon.csv",
        pandas_kwargs= {'usecols': columns_to_keep}
    )
    
    # 2. Convert downloaded file from CSV to JSON format
    pokemon_data = df.to_dict(orient='records')
    
    # 3. Removing any leading or trailing spaces to avoid confusion later on
    df.columns = df.columns.str.strip()
    
    # 4. Save to our local folder
    os.makedirs('../data/raw', exist_ok=True) # Creates the folders if they don't exist yet
    with open(LOCAL_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(pokemon_data, f, indent=4)
        
    print(f"Download complete. Saved locally to {LOCAL_JSON_PATH}")
else:
    print("Local file Kaggle file found.")

# 5. read the downloaded file
df = pd.read_json(LOCAL_JSON_PATH)

print("\nData is ready for processing")
print(f"Shape: {df.shape}")
df.head()

#### Phase 2: Data Profiling

In [ ]:
# Structure Check
print(df.info())

# Checking for missing values
missing_values = df.isna().sum()
print(f"Amount of missing values: \n\n{missing_values}")

# Validity and Anomaly check
type1_col = [col for col in df.columns if 'Type1' in col][0]
print(f"\n Messy formatting detected in '{type1_col}':")
print(df[type1_col].unique()) # This displays all unique typings in Column 'Type1'

print(f"\n Duplicate rows check: {df.duplicated().sum()} found")

#### Phase 3: Data Quality Rules (The Contract)
Based on the profiling above, the following transformation rules will be applied and documented for the Data Engineering team:
1. **Structure:** Strip all leading/trailing whitespace from column names.
2. **Standardization:** Strip whitespace and apply Title Case to all `Type` columns (e.g., `"GRASS "` → `"Grass"`).
3. **Completeness:** Fill missing `Type2` values with `"None"` (since many Pokémon are single-type).
4. **Validity:** Ensure all stat columns (`HP`, `Attack`, `Defense`, `Base_Total`) are strictly numeric.
5. **Uniqueness:** Drop any duplicate Pokémon entries based on `Name`.